In [21]:
import requests
from minsearch import Index
def load_faq_data():
    docs_url = 'https://datatalks.club/faq/json/courses.json'
    response = requests.get(docs_url)
    courses_raw = response.json()

    documents = []
    url_prefix = 'https://datatalks.club/faq'

    for course in courses_raw:
        course_url = f'{url_prefix}{course["path"]}'
        course_response = requests.get(course_url)
        course_response.raise_for_status()
        course_data = course_response.json()

        documents.extend(course_data)

    return documents

In [22]:
documents = load_faq_data()
print(documents[0].keys())

dict_keys(['id', 'course', 'section', 'question', 'answer'])


In [5]:
def build_index(documents):
    index = Index(
        text_fields=['question', 'section', 'answer'],
        keyword_fields=['course']
    )
    index.fit(documents)
    return index

In [6]:
doc = load_faq_data()
index = build_index(doc)
question = "Can I still join the course?"
def search(index, question, course="llm-zoomcamp"):
    boost = {'question': 2.0, 'section': 0.5}
    filter_dict = {"course": course}

    results = index.search(
        query=question,
        filter_dict=filter_dict,
        boost_dict=boost,
        num_results=5,
        output_ids=True
    )

    return results

In [7]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()
search_results = search(index, question)
context = build_context(search_results)

In [8]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know.
"""
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [9]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()
prompt = build_prompt(question, search_results)

In [14]:
from openai import OpenAI
openai_client = OpenAI()
def llm(instructions, user_prompt, model="gpt-5.4-mini"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text

In [15]:
response = llm(INSTRUCTIONS, prompt)
print(response)

Yes, you can still join the course. If you want to receive a certificate, you need to submit your project while the course is still accepting submissions.


In [13]:
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

cost

0.0007830000000000001

In [16]:
from dotenv import load_dotenv
load_dotenv()

from ingest import load_faq_data, build_index
from rag_helper import RAGBase
from openai import OpenAI

documents = load_faq_data()
index = build_index(documents)

openai_client = OpenAI()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

answer = assistant.rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can join now. If you want a certificate, make sure to submit your project while submissions are still open.


In [17]:
custom_instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=custom_instructions,
)
assistant.rag("How do I get a certificate?")

'You can get a certificate only if you finish the course with a **live cohort** and **pass the Capstone project**.\n\nA few important points:\n- **Self-paced mode does not award certificates**\n- You need to **peer-review 3 capstones** after submitting your project\n- Certificates are only available while the course is running, since the peer-review list is compiled then\n\nAlso, if you don’t want the default name on your certificate, make sure your **official name** is filled in on your course profile.'

In [19]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()
documents_n = []

for file in files:
    doc = file.parse()
    documents_n.append(doc)
print(len(documents_n))

72


In [ ]:

index_n = build_index(documents_n)
assistant = RAGBase(
    index=index_n,
    llm_client=openai_client,
)

answer = assistant.rag("How does the agentic loop keep calling the model until it stops?")
print(answer)

I don't know.
